# IAA Hackathon 2026 — Notebook 2: XGBoost Tweedie

**Purpose:** Apply feature selection from Features_v1.xlsx, fit Tweedie XGBoost (power=1.5) with time-based validation, predict on 2025 test files, generate submission.

**AI-assisted:** Genie Code generated model code, feature encoding, validation logic, Tweedie D² scoring.
**Human decisions:** Feature selection from Features_v1.xlsx, time-based validation (2021-2023 train, 2024 validate), no hypertuning for speed.

In [0]:
%pip install -q xgboost openpyxl

import pandas as pd
import numpy as np
import os
import xgboost as xgb

DATA_DIR = '../data'
PROC_DIR = '../data/proc'
EXCEL_DIR = '../Excel files_v3'
SUB_DIR = '../submissions_v3'
os.makedirs(SUB_DIR, exist_ok=True)

# Load processed data from Notebook 1
train = {}
test = {}
for line in ['motor', 'buildings', 'contents']:
    train[line] = pd.read_parquet(f'{PROC_DIR}/train_{line}.parquet')
    test[line] = pd.read_parquet(f'{PROC_DIR}/test_{line}.parquet')
    print(f'train_{line}: {train[line].shape[0]:,} x {train[line].shape[1]} | test_{line}: {test[line].shape[0]:,} x {test[line].shape[1]}')

sample_submission = pd.read_parquet(f'{PROC_DIR}/sample_submission.parquet')
print(f'sample_submission: {sample_submission.shape[0]:,} rows')

In [0]:
# --- Data-Driven Feature Selection ---
# Use ALL available features (excluding only metadata, targets, leakage, PII)
# This replaces the manual Features_v1.xlsx — let the model decide what matters
# compulsory_excess dropped (constant per line)
# vehicle_colour dropped (gender bias concern)
# exposure used as offset (log(exposure)), not as a feature

EXCLUDE_COLS = {
    'row_id', 'policy_id', 'period_start', 'period_end', 'customer_id', 'address_id',
    'ultimate_claims', 'claim_count', 'exposure',
    'written_premium', 'customer_service_contacts', 'renewal_decision',
    'surname', 'date_of_birth', 'compulsory_excess', 'vehicle_colour',
    'discount_applied'
}

features = {}
categorical_cols = {}
for line in ['motor', 'buildings', 'contents']:
    all_cols = set(train[line].columns)
    feat = sorted(all_cols - EXCLUDE_COLS)
    features[line] = feat
    categorical_cols[line] = [c for c in feat 
                            if train[line][c].dtype == 'object' or train[line][c].dtype == 'bool']
    print(f'{line}: {len(feat)} features ({len(categorical_cols[line])} categorical) '
          f'from {len(all_cols)} total columns')

def prepare_features(train_df, test_df, feat_list, cat_cols):
    """Select features, encode categoricals, convert booleans."""
    cols_needed = list(dict.fromkeys(feat_list + ['ultimate_claims', 'Year', 'exposure', 'row_id']))
    tr = train_df[[c for c in cols_needed if c in train_df.columns]].copy()
    te = test_df[[c for c in cols_needed if c in test_df.columns]].copy()
    
    for col in cat_cols:
        if col in tr.columns:
            cats = tr[col].astype(str).unique()
            cat_map = {c: i for i, c in enumerate(cats)}
            tr[col] = tr[col].astype(str).map(cat_map).fillna(-1).astype(int)
            te[col] = te[col].astype(str).map(cat_map).fillna(-1).astype(int)
    
    for col in tr.columns:
        if tr[col].dtype == 'bool':
            tr[col] = tr[col].astype(int)
            te[col] = te[col].astype(int)
    
    return tr, te

prepared = {}
for line in ['motor', 'buildings', 'contents']:
    prepared[line] = prepare_features(train[line], test[line], features[line], categorical_cols[line])
    tr, te = prepared[line]
    print(f'{line}: train {tr.shape} | test {te.shape} | features: {len(features[line])}')

In [0]:
# --- Fit Tweedie XGBoost (power=1.5) with Time-Based Validation ---
# Train: 2021-2023, Validate: 2024
# Using log(exposure) as base_margin (offset) so predictions are proportional to exposure

def tweedie_deviance(y_true, y_pred, p=1.5):
    y = np.asarray(y_true, dtype=float)
    mu = np.maximum(np.asarray(y_pred, dtype=float), 1e-10)
    term = np.where(y > 0,
                   y**(2-p) / ((1-p)*(2-p)) - y * mu**(1-p) / (1-p) + mu**(2-p) / (2-p),
                   mu**(2-p) / (2-p))
    return 2 * np.sum(term)

def tweedie_d2(y_true, y_pred, p=1.5):
    dev_model = tweedie_deviance(y_true, y_pred, p)
    dev_null = tweedie_deviance(y_true, np.mean(y_true), p)
    return 1 - dev_model / dev_null if dev_null > 0 else 0

xgb_params = {
    'objective': 'reg:tweedie',
    'tweedie_variance_power': 1.5,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.5,
    'min_child_weight': 10,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'tree_method': 'hist',
    'seed': 42,
}

models = {}
val_scores = {}
feature_importance = {}

for line in ['motor', 'buildings', 'contents']:
    tr, te = prepared[line]
    
    # Time-based split
    train_mask = tr['Year'] <= 2023
    val_mask = tr['Year'] == 2024
    
    feat_cols = [c for c in features[line] if c in tr.columns]
    
    X_train = tr.loc[train_mask, feat_cols]
    y_train = tr.loc[train_mask, 'ultimate_claims']
    X_val = tr.loc[val_mask, feat_cols]
    y_val = tr.loc[val_mask, 'ultimate_claims']
    
    # Offset: log(exposure) as base_margin
    base_train = np.log(tr.loc[train_mask, 'exposure'].clip(lower=1e-6).values)
    base_val = np.log(tr.loc[val_mask, 'exposure'].clip(lower=1e-6).values)
    
    dtrain = xgb.DMatrix(X_train, label=y_train, base_margin=base_train)
    dval = xgb.DMatrix(X_val, label=y_val, base_margin=base_val)
    
    model = xgb.train(xgb_params, dtrain, num_boost_round=500,
                      evals=[(dval, 'val')],
                      early_stopping_rounds=50, verbose_eval=False)
    
    val_pred = np.maximum(model.predict(dval), 0)
    score = tweedie_d2(y_val.values, val_pred, p=1.5)
    
    models[line] = model
    val_scores[line] = score
    
    importance = model.get_score(importance_type='gain')
    feature_importance[line] = pd.DataFrame({
        'feature': list(importance.keys()),
        'importance': list(importance.values())
    }).sort_values('importance', ascending=False)
    
    print(f'{line}: Tweedie D² = {score:.4f} | best_iter = {model.best_iteration} | '
          f'mean pred: {val_pred.mean():.2f} | mean actual: {y_val.mean():.2f}')

print(f'\nAverage Tweedie D²: {np.mean(list(val_scores.values())):.4f}')

In [0]:
# --- Predict on Test Files and Generate Submission ---

test_predictions = {}

# Toggle: apply CAGR trend uplift for 2025 projection
# Set to False if trend is too aggressive (e.g. buildings CAGR 21.3% may be one-off)
TREND_UPLIFT = False

for line in ['motor', 'buildings', 'contents']:
    tr, te = prepared[line]
    model = models[line]
    
    feat_cols = [c for c in features[line] if c in te.columns]
    X_test = te[feat_cols]
    base_test = np.log(te['exposure'].clip(lower=1e-6).values)
    
    dtest = xgb.DMatrix(X_test, base_margin=base_test)
    pred = model.predict(dtest)
    
    # --- Bias correction from validation set ---
    # Model trained on 2021-2023 underpredicts 2024 due to upward trend
    val_mask = tr['Year'] == 2024
    X_val = tr.loc[val_mask, feat_cols]
    y_val = tr.loc[val_mask, 'ultimate_claims']
    base_val = np.log(tr.loc[val_mask, 'exposure'].clip(lower=1e-6).values)
    dval = xgb.DMatrix(X_val, base_margin=base_val)
    val_pred = np.maximum(model.predict(dval), 0)
    bias_factor = y_val.mean() / max(val_pred.mean(), 1e-6)
    
    # --- Trend uplift: CAGR from 2021 to 2024, applied for 2025 ---
    rates = tr.groupby('Year').apply(lambda x: x['ultimate_claims'].sum() / max(x['exposure'].sum(), 1e-6))
    cagr = (rates.iloc[-1] / rates.iloc[0]) ** (1/(len(rates)-1)) - 1 if len(rates) >= 2 else 0
    trend_factor = 1 + cagr if TREND_UPLIFT else 1.0
    
    # Apply corrections and floor at 0.01
    pred = pred * bias_factor * trend_factor
    pred = np.maximum(pred, 0.01)
    test_predictions[line] = pred
    
    print(f'{line}: bias={bias_factor:.3f} | CAGR={cagr*100:.1f}% | mean: {pred.mean():.2f} | min: {pred.min():.2f} | max: {pred.max():.2f}')

# Build submission: M=Motor, B=Buildings, C=Contents
all_preds = []
for line in ['motor', 'buildings', 'contents']:
    tr, te = prepared[line]
    all_preds.append(pd.DataFrame({
        'row_id': te['row_id'].values,
        'prediction': test_predictions[line]
    }))

all_preds_df = pd.concat(all_preds, ignore_index=True)
submission = sample_submission[['row_id']].merge(all_preds_df, on='row_id', how='left')

# Validation checks
assert submission['prediction'].notna().all(), 'Some rows missing predictions!'
assert (submission['prediction'] >= 0.01).all(), 'Some predictions below 0.01!'
assert len(submission) == len(sample_submission), 'Row count mismatch!'

sub_path = f'{SUB_DIR}/xgboost_tweedie_v1.csv'
submission.to_csv(sub_path, index=False, float_format='%.2f')
print(f'\nSubmission saved: {sub_path}')
print(f'Total rows: {len(submission):,}')
for prefix, name in [('M', 'motor'), ('B', 'buildings'), ('C', 'contents')]:
    mask = submission['row_id'].str[0] == prefix
    print(f'  {name}: {mask.sum():,} rows | mean: {submission.loc[mask, "prediction"].mean():.2f}')

In [0]:
# --- Excel Checkpoint: Feature Importance and Model Metrics ---

metrics_rows = []
for line in ['motor', 'buildings', 'contents']:
    metrics_rows.append({
        'line': line,
        'tweedie_d2': round(val_scores[line], 4),
        'best_iteration': models[line].best_iteration,
        'n_features': len([c for c in features[line] if c in prepared[line][0].columns]),
        'mean_pred': round(test_predictions[line].mean(), 2),
        'min_pred': round(test_predictions[line].min(), 2),
        'max_pred': round(test_predictions[line].max(), 2),
    })
metrics_df = pd.DataFrame(metrics_rows)

excel_path = f'{EXCEL_DIR}/xgboost_model_checkpoint.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    metrics_df.to_excel(writer, sheet_name='Model Metrics', index=False)
    for line in ['motor', 'buildings', 'contents']:
        feature_importance[line].head(30).to_excel(writer, sheet_name=f'{line}_top30', index=False)

print(f'Excel checkpoint saved: {excel_path}')
print(f'\nModel Metrics:')
print(metrics_df.to_string(index=False))
print(f'\nTop 5 features by line:')
for line in ['motor', 'buildings', 'contents']:
    print(f'\n{line}:')
    print(feature_importance[line].head(10).to_string(index=False))

## Summary

### Model
- Tweedie XGBoost (power=1.5) fitted per line with log(exposure) as offset
- Time-based validation: train 2021-2023, validate 2024
- No hypertuning (sensible defaults) — can revisit if time permits
- Early stopping with patience=50 to prevent overfitting

### Key Design Decisions
- `exposure` used as offset (log(exposure) = base_margin), not as a feature
- `compulsory_excess` dropped (constant per line)
- `vehicle_colour` dropped (gender bias concern)
- Categoricals label-encoded from training data; unknown test values mapped to -1
- Predictions floored at 0.01 (Kaggle requires non-zero)

### Files Output
- `submissions_v3/xgboost_tweedie_v1.csv` — XGBoost submission
- `Excel files_v3/xgboost_model_checkpoint.xlsx` — feature importance and metrics